# Dubly ME — Sequential Colab Pipeline (Phases A–D)

Run this notebook cell-by-cell in Google Colab (GPU runtime, T4 recommended).

**Architecture:**
- Google Drive is mounted for persistent storage of models and artifacts.
- `HF_HOME` is pointed at Drive so heavy LLMs are never re-downloaded.
- Each phase invokes the corresponding CLI script from `src/`.
- All source code and dependencies are pulled from the GitHub repo — no manual patches needed.

## Cell 0 — Environment Setup

Mount Google Drive, configure persistent HuggingFace model cache,
clone/sync the repo, and install Python dependencies.

In [1]:
import os
import subprocess

# ── 0.1  Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── 0.2  Persistent model cache on Drive (prevents re-downloading) ──
#    This MUST be set BEFORE any import of transformers / pyannote / etc.
STORAGE_ROOT = "/content/drive/MyDrive/Dubly_ME_Storage"
os.environ["HF_HOME"]       = f"{STORAGE_ROOT}/models"
os.environ["HF_HUB_CACHE"]  = f"{STORAGE_ROOT}/models/hub"
os.environ["TORCH_HOME"]    = f"{STORAGE_ROOT}/models/torch"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)

print(f"✔ HF_HOME       = {os.environ['HF_HOME']}")
print(f"✔ HF_HUB_CACHE  = {os.environ['HF_HUB_CACHE']}")
print(f"✔ TORCH_HOME    = {os.environ['TORCH_HOME']}")

# ── 0.3  Set HF_TOKEN from Colab Secrets (required for pyannote) ────
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✔ HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("⚠ Could not load HF_TOKEN from Colab Secrets.")
    print("  Set it manually:  os.environ['HF_TOKEN'] = 'hf_your_token'")

# ── 0.4  Sync the project repo to the Colab VM (via Git) ──────────
REPO_DIR = "/content/Dubly_ME"
GIT_URL = "https://github.com/Omar-Gemy/Dubly_ME.git"
BRANCH = "feature/upgrade-stack"

if not os.path.exists(REPO_DIR):
    print(f"▶ Cloning repo from {GIT_URL} (branch: {BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO_DIR], check=True)
    print("✔ Repo cloned successfully.")
else:
    print("▶ Pulling latest changes...")
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=REPO_DIR, check=True)
    print("✔ Repo updated successfully.")

# ── 0.5  Install dependencies ──────────────────────────────────
subprocess.run(
    ["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
    check=True,
)
print("\n✔ All dependencies installed.")

# ── 0.6  Verify FFmpeg (pre-installed on Colab GPU runtimes) ───────
subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
print("✔ FFmpeg is available.")

print("\n" + "═" * 60)
print("  ✅  Cell 0: Environment Setup Complete")
print("═" * 60)

Mounted at /content/drive
✔ HF_HOME       = /content/drive/MyDrive/Dubly_ME_Storage/models
✔ HF_HUB_CACHE  = /content/drive/MyDrive/Dubly_ME_Storage/models/hub
✔ TORCH_HOME    = /content/drive/MyDrive/Dubly_ME_Storage/models/torch
✔ HF_TOKEN loaded from Colab Secrets.
▶ Cloning repo from https://github.com/Omar-Gemy/Dubly_ME.git (branch: feature/upgrade-stack)...
✔ Repo cloned successfully.

✔ All dependencies installed.
✔ FFmpeg is available.

════════════════════════════════════════════════════════════
  ✅  Cell 0: Environment Setup Complete
════════════════════════════════════════════════════════════


## Cell 0b — ASR Switchboard (Legacy vs. Experimental)

Toggle which ASR engine Phase C uses, **without touching the legacy path**.

- `ASR_MODE = "legacy"` → existing WhisperX large-v3 + wav2vec2 forced alignment (writes `artifacts/transcripts.json`, required by Phase D).
- `ASR_MODE = "experimental"` → local Egyptian-fine-tuned CTranslate2 model via `faster-whisper`, **transcription only** (no alignment, no `transcripts.json`) — diagnostic output for manual comparison.

Flip `ASR_MODE` back to `"legacy"` at any time to restore the full pipeline.

In [2]:
# ── ASR mode switch: "legacy" | "experimental" ──
ASR_MODE = "legacy"

# Local CTranslate2 model directory for experimental mode (output of
# merge_lora.py + ct2-transformers-converter). Kept on Drive so it survives
# across Colab sessions; change to any local path if you prefer.
EXPERIMENTAL_ASR_MODEL_PATH = "/content/drive/MyDrive/Dubly_ME_Storage/models/whisper-large-v3-egy-ct2"
EXPERIMENTAL_ASR_LANGUAGE = "ar"
EXPERIMENTAL_ASR_DEVICE = "cuda"
EXPERIMENTAL_ASR_COMPUTE_TYPE = "float16"

print(f"✔ ASR switchboard set: ASR_MODE = {ASR_MODE!r}")
if ASR_MODE == "experimental":
    print(f"  experimental model: {EXPERIMENTAL_ASR_MODEL_PATH}")
    print("  ⚠ experimental = transcription only; Phase D still needs a legacy run.")

✔ ASR switchboard set: ASR_MODE = 'legacy'


## Cell 0c — Model Conversion (Prepare Experimental ASR)

Build the experimental Egyptian **CTranslate2** model in-place inside Colab.
Idempotent — only builds what is missing:

1. CT2 model dir already exists → **skip**.
2. Merged HF checkpoint missing → run `merge_lora.py` (LoRA merge).
3. Merged exists but CT2 missing → run `ct2-transformers-converter`.

Only needed when you intend to run Phase C with `ASR_MODE = "experimental"`.
Safe to run in legacy mode too — it just reports "already present / skipping".

In [3]:
import os, subprocess

REPO_DIR = "/content/Dubly_ME"

# CT2 dir that experimental mode loads (defined in the Cell 0b switchboard),
# so we convert to exactly the path Phase C will read from.
CT2_MODEL_DIR = EXPERIMENTAL_ASR_MODEL_PATH
# merge_lora.py writes its merged HF checkpoint here (OUT is relative to its cwd).
MERGED_DIR = f"{REPO_DIR}/whisper-large-v3-egy-merged"

if os.path.isdir(CT2_MODEL_DIR):
    print(f"✔ CT2 model already present — skipping conversion:\n    {CT2_MODEL_DIR}")
else:
    print(f"▶ CT2 model not found at:\n    {CT2_MODEL_DIR}\n  Building it now.")

    # merge_lora.py needs peft (absent from requirements-colab.txt). ctranslate2's
    # converter comes transitively with whisperx, so its version matches runtime.
    print("▶ Ensuring conversion deps (peft, torchao) are upgraded…")
    subprocess.run(["pip", "install", "-q", "peft", "torchao>=0.16.0"], check=True)

    # ── Step 1: LoRA merge (only if the merged folder is missing) ──
    if os.path.isdir(MERGED_DIR):
        print(f"✔ Merged HF model already present — skipping merge:\n    {MERGED_DIR}")
    else:
        print("▶ Running merge_lora.py (LoRA merge → merged HF checkpoint)…")
        subprocess.run(["python", f"{REPO_DIR}/merge_lora.py"], check=True, cwd=REPO_DIR)

    # ── Step 2: CTranslate2 conversion ──
    print(f"▶ Converting → CTranslate2:\n    {MERGED_DIR}\n    → {CT2_MODEL_DIR}")
    os.makedirs(os.path.dirname(CT2_MODEL_DIR), exist_ok=True)
    subprocess.run(
        [
            "ct2-transformers-converter",
            "--model", MERGED_DIR,
            "--output_dir", CT2_MODEL_DIR,
            "--copy_files", "tokenizer.json", "preprocessor_config.json",
            "--quantization", "float16",
            "--force",
        ],
        check=True,
    )
    print(f"✔ Conversion complete — CT2 model ready at:\n    {CT2_MODEL_DIR}")

print("\n" + "═" * 60)
print("  ✅  Cell 0c: Model Conversion check complete")
print("═" * 60)

✔ CT2 model already present — skipping conversion:
    /content/drive/MyDrive/Dubly_ME_Storage/models/whisper-large-v3-egy-ct2

════════════════════════════════════════════════════════════
  ✅  Cell 0c: Model Conversion check complete
════════════════════════════════════════════════════════════


In [ ]:
###
import os
import subprocess

REPO_DIR = "/content/Dubly_ME"
MERGED_DIR = f"{REPO_DIR}/whisper-large-v3-egy-merged"

# 1. التأكد أولاً من وجود الملف نفسه في الكولاب قبل أي تشغيل
script_path = f"{REPO_DIR}/merge_lora.py"
if not os.path.exists(script_path):
    print(f"❌ Error: {script_path} does not exist! Did you forget to git push it from your local machine?")
else:
    print(f"✔ Found merge_lora.py at {script_path}")
    print("▶ Ensuring peft is installed...")
    subprocess.run(["pip", "install", "-q", "peft"], check=True)

    print("▶ Running merge_lora.py and capturing full output...")
    # هنا قمنا بإلغاء إخفاء المخرجات ونلتقط الـ stdout والـ stderr لعرضهم بالكامل في الكولاب
    result = subprocess.run(
        ["python", script_path],
        cwd=REPO_DIR,
        text=True,
        capture_output=True
    )

    print("\n" + "="*40 + " SCRIPT STDOUT " + "="*40)
    print(result.stdout if result.stdout else "(No stdout output)")

    print("\n" + "="*40 + " SCRIPT STDERR (THE REAL ERROR) " + "="*40)
    print(result.stderr if result.stderr else "(No stderr output)")

    if result.returncode != 0:
        print(f"\n❌ Script failed with exit code {result.returncode}")
    else:
        print("\n✔ Script executed successfully successfully!")

✔ Found merge_lora.py at /content/Dubly_ME/merge_lora.py
▶ Ensuring peft is installed...
▶ Running merge_lora.py and capturing full output...

======================================== SCRIPT STDOUT ========================================
▶ Loading base model: openai/whisper-large-v3
▶ Applying + merging adapter: AbdelrahmanHassan/whisper-large-v3-egyptian-arabic


======================================== SCRIPT STDERR (THE REAL ERROR) ========================================
2026-07-05 22:50:23.705919: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/Dubly_ME/merge_lora.py", line 66, in <module>
    main()
  File "/content/Dubly_ME/merge_lora.py", line 48, in main
    peft_model = PeftModel.from

## Cell 1 — Phase A: Audio Ingestion & Voice Activity Detection

Extract audio from the source media file, normalise loudness (EBU R128),
and run Silero VAD to detect speech segments.

- **Script:** `src/ingestion_vad.py`
- **Input:** source media file (video or audio)
- **Output:** `artifacts/segments.json` + `data/audio_out/_temp_normalised.wav`

In [4]:
import subprocess, os

REPO_DIR = "/content/Dubly_ME"
INPUT_MEDIA = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"

# Validate input exists
if not os.path.isfile(INPUT_MEDIA):
    raise FileNotFoundError(
        f"Source media not found: {INPUT_MEDIA}\n"
        f"Upload your video/audio file to this Google Drive path."
    )

print(f"▶ Phase A: Ingesting {INPUT_MEDIA}")
subprocess.run(
    [
        "python", f"{REPO_DIR}/src/ingestion_vad.py",
        INPUT_MEDIA,
        "--source-lang", "ar",
        "--target-lang", "en",
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase A: Audio Ingestion & VAD Complete")
print("═" * 60)

▶ Phase A: Ingesting /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4

════════════════════════════════════════════════════════════
  ✅  Phase A: Audio Ingestion & VAD Complete
════════════════════════════════════════════════════════════


## Cell 2 — Phase B: Speaker Diarization

Assign speaker identities to VAD segments using the pyannote.audio 4.x
ensemble (Segmentation → Embedding → Clustering).

- **Script:** `src/diarization.py`
- **Input:** `data/audio_out/_temp_normalised.wav` + `artifacts/segments.json`
- **Output:** `artifacts/segments.json` (enriched with `speaker_id`)
- **Requires:** `HF_TOKEN` set (for gated pyannote models).

In [5]:
import subprocess

REPO_DIR = "/content/Dubly_ME"

print("▶ Phase B: Speaker Diarization")
subprocess.run(
    [
        "python", f"{REPO_DIR}/src/diarization.py",
        "--device", "cuda",
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase B: Speaker Diarization Complete")
print("═" * 60)

▶ Phase B: Speaker Diarization

════════════════════════════════════════════════════════════
  ✅  Phase B: Speaker Diarization Complete
════════════════════════════════════════════════════════════


## Cell 3 — Phase C: ASR Transcription

Transcribe the full audio in one pass with WhisperX (large-v3 backend),
run wav2vec2 forced alignment for precise word-level timestamps, then
intersect words with the diarization segments (global-to-local mapping).

- **Script:** `src/asr_transcription.py`
- **Input:** `data/audio_out/_temp_normalised.wav` + `artifacts/segments.json`
- **Output:** `artifacts/transcripts.json`

In [6]:
import subprocess, os

REPO_DIR = "/content/Dubly_ME"


def run_experimental_asr(
    model_path,
    audio_path,
    language="ar",
    device="cuda",
    compute_type="float16",
    beam_size=5,
):
    """
    [EXPERIMENTAL] Transcribe the normalised WAV directly with a local
    CTranslate2 model (Egyptian fine-tune) via faster-whisper.

    Transcription ONLY — no wav2vec2 forced alignment and no diarization
    intersection, so this does NOT write artifacts/transcripts.json. It prints
    the transcript for manual comparison against the legacy WhisperX output;
    Phase D still requires a legacy ASR run to produce its input contract.
    """
    from faster_whisper import WhisperModel

    # CTranslate2 has no fp16 CPU kernels — mirror src/asr_transcription.py.
    if device == "cpu" and compute_type == "float16":
        compute_type = "int8"
        print("  ⚠ float16 unsupported on CPU — using int8 compute type")

    if not os.path.isdir(model_path):
        raise FileNotFoundError(
            f"Experimental CT2 model dir not found: {model_path}\n"
            f"Run merge_lora.py + ct2-transformers-converter first, or point "
            f"EXPERIMENTAL_ASR_MODEL_PATH at the converted model directory."
        )

    print(f"▶ [experimental] Loading CT2 model: {model_path}")
    print(f"  device={device}  compute_type={compute_type}")
    model = WhisperModel(model_path, device=device, compute_type=compute_type)

    print("▶ [experimental] Transcribing (transcription only, NO alignment)…")
    segments, info = model.transcribe(audio_path, language=language, beam_size=beam_size)

    print(f"  language: {info.language}")
    print("═" * 60)
    for seg in segments:
        print(f"[{seg.start:7.2f}s → {seg.end:7.2f}s]  {seg.text.strip()}")
    print("═" * 60)
    print("  ⚠ Diagnostic only — no transcripts.json written.")
    print("    Set ASR_MODE = 'legacy' to produce the Phase D contract.")


print(f"▶ Phase C: ASR Transcription  [ASR_MODE = {ASR_MODE}]")

if ASR_MODE == "legacy":
    # ── LEGACY PATH (unchanged): WhisperX large-v3 + wav2vec2 forced alignment ──
    print("▶ Phase C: ASR Transcription")
    subprocess.run(
        [
            "python", f"{REPO_DIR}/src/asr_transcription.py",
            "--model", "large-v3",
            "--source-lang", "ar",
            "--device", "cuda",
            "--compute-type", "float16",
        ],
        check=True,
        cwd=REPO_DIR,
    )

elif ASR_MODE == "experimental":
    # ── EXPERIMENTAL PATH: local Egyptian CT2 model, transcription only ──
    run_experimental_asr(
        model_path=EXPERIMENTAL_ASR_MODEL_PATH,
        audio_path=f"{REPO_DIR}/data/audio_out/_temp_normalised.wav",
        language=EXPERIMENTAL_ASR_LANGUAGE,
        device=EXPERIMENTAL_ASR_DEVICE,
        compute_type=EXPERIMENTAL_ASR_COMPUTE_TYPE,
    )

else:
    raise ValueError(
        f"Unknown ASR_MODE={ASR_MODE!r} — set 'legacy' or 'experimental' "
        f"in the [Cell 0b] switchboard."
    )

print("\n" + "═" * 60)
print("  ✅  Phase C: ASR Transcription Complete")
print("═" * 60)

▶ Phase C: ASR Transcription  [ASR_MODE = legacy]
▶ Phase C: ASR Transcription

════════════════════════════════════════════════════════════
  ✅  Phase C: ASR Transcription Complete
════════════════════════════════════════════════════════════


## Cell 4 — Phase D: Contextual Translation (Isochronous)

Adapt the Arabic transcripts into natural spoken English for dubbing using
Qwen2.5-14B-Instruct-AWQ. The model is a pre-quantized **4-bit AWQ** checkpoint
(~9–10 GB VRAM, fits the 16 GB T4) and is cached on Google Drive via `HF_HOME`
to prevent re-downloading on subsequent sessions.

Each line is adapted against an explicit **isochrony budget** (target syllable
count derived from the segment duration) so the dub fits its time window.

- **Script:** `src/translation.py`
- **Input:** `artifacts/transcripts.json`
- **Output:** `artifacts/translation.json`

In [7]:
import subprocess

REPO_DIR = "/content/Dubly_ME"

print("▶ Phase D: Contextual Translation (Isochronous)")
subprocess.run(
    [
        "python", f"{REPO_DIR}/src/translation.py",
        "--model", "Qwen/Qwen2.5-14B-Instruct-AWQ",
        "--device", "auto",
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase D: Translation Complete")
print("═" * 60)

▶ Phase D: Contextual Translation (Isochronous)

════════════════════════════════════════════════════════════
  ✅  Phase D: Translation Complete
════════════════════════════════════════════════════════════


## Cell 5 — Pipeline Summary: Verify Artifacts

Quick sanity check: verify that all expected output artifacts exist
and print a summary of what was produced.

In [8]:
import json, os

REPO_DIR = "/content/Dubly_ME"
ARTIFACTS = f"{REPO_DIR}/artifacts"

expected_files = {
    "Phase A (VAD)":         f"{ARTIFACTS}/segments.json",
    "Phase C (ASR)":         f"{ARTIFACTS}/transcripts.json",
    "Phase D (Translation)": f"{ARTIFACTS}/translation.json",
}

print("Pipeline Artifact Verification")
print("─" * 50)
all_ok = True
for phase, path in expected_files.items():
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        # Load to count segments
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        n_segs = data.get("total_segments", len(data.get("segments", [])))
        print(f"  ✔ {phase:<25s}  {n_segs:>3d} segments  ({size_kb:.1f} KB)")
    else:
        print(f"  ✖ {phase:<25s}  FILE MISSING: {path}")
        all_ok = False

print("─" * 50)
if all_ok:
    print("  ✅  All pipeline artifacts verified successfully!")
else:
    print("  ⚠  Some artifacts are missing — check the logs above.")
print("═" * 50)

Pipeline Artifact Verification
──────────────────────────────────────────────────
  ✔ Phase A (VAD)               44 segments  (8.2 KB)
  ✔ Phase C (ASR)               44 segments  (15.4 KB)
  ✔ Phase D (Translation)       44 segments  (29.7 KB)
──────────────────────────────────────────────────
  ✅  All pipeline artifacts verified successfully!
══════════════════════════════════════════════════


## Cell 6 — Phase E: Voice Cloning & TTS Synthesis (XTTS v2)

Synthesize English dubbed audio for each translated segment using
self-hosted **XTTS v2** (Coqui), cloning the speaker's voice from a
reference clip.

**Dependency isolation (important).** Coqui XTTS v2 requires a `transformers`
build that is *incompatible* with the pinned `transformers==4.47.1` used by
Phases C/D (WhisperX + AutoAWQ): the classic `TTS` package needs
`transformers<4.41`, while `coqui-tts>=0.27` needs `transformers>=4.57`.
Neither can share the A-D pin. Phase E therefore runs inside its **own
virtualenv** (`.venv_tts`) with its own dependency set - the A-D kernel is
never mutated, so Phases A-D remain re-runnable at any time.

- **Script:** `src/tts_synthesis.py` (unchanged - invoked via the venv's Python)
- **Input:** `artifacts/translation.json` + a voice reference
  (`artifacts/voice_ref.wav`, or auto-extracted from the source video)
- **Output:** `artifacts/audio_out/segment_XXX.wav` + `artifacts/tts_manifest.json`

In [ ]:
# -- Phase E setup: isolated virtualenv for Coqui XTTS v2 --------------
# XTTS needs a transformers build that conflicts with the A-D pin
# (transformers==4.47.1). We build a dedicated venv so the two never mix.
import os, subprocess, sys

VENV_TTS = "/content/.venv_tts"
VENV_PY  = f"{VENV_TTS}/bin/python"


def _tts_env_ready():
    """True if the venv exists and can import the TTS runtime deps."""
    if not os.path.exists(VENV_PY):
        return False
    probe = subprocess.run(
        [VENV_PY, "-c", "import TTS.api, torch, soundfile"],
        capture_output=True,
    )
    return probe.returncode == 0


if _tts_env_ready():
    print(f"✔ TTS venv already present - skipping build:\n    {VENV_TTS}")
else:
    print(f"▶ Building isolated TTS venv at {VENV_TTS} ...")
    subprocess.run([sys.executable, "-m", "venv", VENV_TTS], check=True)
    subprocess.run(
        [VENV_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True
    )

    # torch first (coqui-tts >=0.27 does NOT pull torch itself), then the
    # maintained Coqui fork (provides TTS.api / XTTS v2) + audio I/O.
    print("▶ Installing torch (CUDA build) into the TTS venv ...")
    subprocess.run([VENV_PY, "-m", "pip", "install", "-q", "torch"], check=True)
    print("▶ Installing coqui-tts + soundfile into the TTS venv ...")
    subprocess.run(
        [VENV_PY, "-m", "pip", "install", "-q", "coqui-tts", "soundfile"],
        check=True,
    )

    if not _tts_env_ready():
        raise RuntimeError(
            "TTS venv build finished but 'import TTS.api, torch, soundfile' "
            "failed - check the pip logs above."
        )
    print("✔ TTS venv ready.")

print("\n" + "═" * 60)
print("  ✅  Phase E setup: isolated TTS venv ready")
print("═" * 60)

In [ ]:
# -- Phase E: run XTTS v2 synthesis inside the isolated venv -----------
import subprocess, os

REPO_DIR = "/content/Dubly_ME"
VENV_PY  = "/content/.venv_tts/bin/python"

# INPUT_MEDIA is defined in the Phase A cell; fall back to the same default
# path if this cell is run standalone.
INPUT_MEDIA = globals().get(
    "INPUT_MEDIA",
    "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4",
)

print("▶ Phase E: Voice Cloning & TTS (XTTS v2, isolated venv)")
# --auto-extract pulls a fallback voice reference from the source video when
# artifacts/voice_ref.wav is absent. Provide voice_ref.wav yourself for a
# cleaner, hand-picked clone.
subprocess.run(
    [
        VENV_PY, f"{REPO_DIR}/src/tts_synthesis.py",
        "--source", INPUT_MEDIA,
        "--auto-extract",
        "--device", "auto",
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase E: TTS Synthesis Complete")
print("═" * 60)

## Cell 7 — Phase F (Step 1): Duration Fitting (Time-Stretch)

Fit each synthesized segment into its original time window using WSOLA /
FFmpeg `atempo` time-stretching. Tiers: <=1.15x trivial, 1.15-2.0x stretch,
>2.0x capped at 2.0x (overflow allowed to preserve intelligibility).

Runs in the **main kernel** - only needs numpy/soundfile/FFmpeg, no TTS deps.

- **Script:** `src/time_stretch.py`
- **Input:** `artifacts/tts_manifest.json` + `artifacts/audio_out/*.wav`
- **Output:** `artifacts/audio_out/stretched/*.wav` + `artifacts/stretch_manifest.json`

In [ ]:
import subprocess

REPO_DIR = "/content/Dubly_ME"

print("▶ Phase F (Step 1): Duration Fitting (Time-Stretch)")
subprocess.run(
    [
        "python", f"{REPO_DIR}/src/time_stretch.py",
        "--max-ratio", "2.0",
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase F (Step 1): Time-Stretch Complete")
print("═" * 60)

## Cell 8 — Phase F (Step 2): Final Audio Mix & Video Render

Place each time-fitted segment at its original `start_time`, crossfade
boundaries, pass through the original Arabic audio for skipped segments, and
mux the dubbed track over the source video (AAC 192k) via FFmpeg.

- **Script:** `src/mix_render.py`
- **Input:** `artifacts/stretch_manifest.json` + `artifacts/segments.json` + source video
- **Output:** `data/audio_out/final_dubbed.{wav,mp4}` + `artifacts/mix_manifest.json`

In [ ]:
import subprocess, os

REPO_DIR = "/content/Dubly_ME"

# Reuse the same source video the rest of the pipeline ran on.
INPUT_MEDIA = globals().get(
    "INPUT_MEDIA",
    "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4",
)

print("▶ Phase F (Step 2): Final Audio Mix & Video Render")
subprocess.run(
    [
        "python", f"{REPO_DIR}/src/mix_render.py",
        "--source", INPUT_MEDIA,
    ],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase F (Step 2): Mix & Render Complete")
print("═" * 60)

## Cell 9 — Phase F (Step 3): QA Report

Generate the final QA report (timing fit, overflow, skip reasons, mix stats)
from the TTS / stretch / mix manifests + `segments.json`, then print the
human-readable summary inline.

- **Script:** `src/qa_report.py`
- **Output:** `artifacts/qa_report.{json,md}`

In [ ]:
import subprocess, os

REPO_DIR = "/content/Dubly_ME"

print("▶ Phase F (Step 3): QA Report")
subprocess.run(
    ["python", f"{REPO_DIR}/src/qa_report.py"],
    check=True,
    cwd=REPO_DIR,
)

print("\n" + "═" * 60)
print("  ✅  Phase F (Step 3): QA Report Complete")
print("═" * 60)

# Show the human-readable QA summary inline.
qa_md = f"{REPO_DIR}/artifacts/qa_report.md"
if os.path.isfile(qa_md):
    with open(qa_md, "r", encoding="utf-8") as fh:
        print("\n" + fh.read())